In [1]:
import torch
import warnings
from constants import *
from pathlib import Path
from torch import nn, optim
from resnet20 import resnet20
import torch.nn.functional as F
import json, random, numpy as np
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR100
warnings.filterwarnings("ignore")

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
MEAN = (0.5071, 0.4865, 0.4409)
STD  = (0.2673, 0.2564, 0.2762)

train_transforms = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD)
    ]
)
 

test_transforms = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD)
    ]
)

In [4]:
train_dataset = CIFAR100('./res_data/', train=True, download=False, transform=train_transforms)
test_dataset = CIFAR100('./res_data/', train=False, download=False, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
test_loader  = DataLoader(test_dataset, batch_size=min(len(test_dataset), 1000), shuffle=False, num_workers=4, pin_memory=True)

### Backbone: Trained on the whole of the dataset

In [5]:
Path('./res_models').mkdir(parents=True, exist_ok=True)
resnet_backbone_model = resnet20()


EPOCHS = 200
resnet_backbone_model = resnet_backbone_model.to(device)

'''sgd with momentum, not adam, adam loses 2-4% on cifar resnets'''
optimizer = optim.SGD(resnet_backbone_model.parameters(),
                      lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)

'''cosine anneal to zero over the full run, no step schedule'''
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    for image, label in loader:
        image, label = image.to(device, non_blocking=True), label.to(device, non_blocking=True)
        pred = model(image)
        correct += (pred.argmax(dim=1)==label).sum().item()
        total += label.size(0)

    return 100 * correct / total



In [8]:
best_acc = 0.0

for epoch in range(EPOCHS):
    '''train mode lets the batchnorm running stats update'''
    resnet_backbone_model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        outputs = resnet_backbone_model(images)
        loss = F.cross_entropy(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    '''step the schedule once per epoch, not per iteration'''
    scheduler.step()

    train_loss = running_loss / len(train_loader)
    test_acc = evaluate(resnet_backbone_model, test_loader)
    print(f'epoch {epoch+1:3d}/{EPOCHS}  loss {train_loss:.4f}  '
          f'test acc {test_acc:.2f}%  lr {scheduler.get_last_lr()[0]:.5f}')

    '''keep only the best model, this is the shared init every expert descends from'''
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save({
            'state_dict': resnet_backbone_model.state_dict(),
            'model_id': 'backbone_a',
            'split_id': 'full',
            'seed': SEED,
            'epoch': epoch + 1,
            'init_group': 'base_a',
            'test_acc': test_acc,
        }, './res_models/backbone_a.pt')

    '''rolling resume file for crash recovery, overwritten not accumulated'''
    if (epoch + 1) % 10 == 0:
        torch.save({
            'state_dict': resnet_backbone_model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'epoch': epoch + 1,
            'best_acc': best_acc,
        }, './res_models/backbone_a_resume.pt')

print(f'done, best test acc {best_acc:.2f}%')

epoch   1/200  loss 3.9256  test acc 14.90%  lr 0.09999
epoch   2/200  loss 3.3139  test acc 21.14%  lr 0.09998
epoch   3/200  loss 2.8783  test acc 27.76%  lr 0.09994
epoch   4/200  loss 2.5659  test acc 32.87%  lr 0.09990
epoch   5/200  loss 2.3446  test acc 32.97%  lr 0.09985
epoch   6/200  loss 2.1877  test acc 39.70%  lr 0.09978
epoch   7/200  loss 2.0743  test acc 37.28%  lr 0.09970
epoch   8/200  loss 1.9868  test acc 37.69%  lr 0.09961
epoch   9/200  loss 1.9157  test acc 40.34%  lr 0.09950
epoch  10/200  loss 1.8749  test acc 41.67%  lr 0.09938
epoch  11/200  loss 1.8296  test acc 44.08%  lr 0.09926
epoch  12/200  loss 1.7991  test acc 47.30%  lr 0.09911
epoch  13/200  loss 1.7680  test acc 38.87%  lr 0.09896
epoch  14/200  loss 1.7512  test acc 45.03%  lr 0.09880
epoch  15/200  loss 1.7119  test acc 48.29%  lr 0.09862
epoch  16/200  loss 1.7084  test acc 46.23%  lr 0.09843
epoch  17/200  loss 1.6838  test acc 46.79%  lr 0.09823
epoch  18/200  loss 1.6757  test acc 45.25%  lr 

In [6]:
def load_backbone_model(path):
    checkpoint = torch.load(path, map_location=device)
    model = resnet20().to(device)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    return model

In [7]:
resnet_backbone_model = load_backbone_model('./res_models/backbone_a.pt')

In [8]:
evaluate(resnet_backbone_model, test_loader)

68.79

In [9]:
! python extract.py ./res_models/backbone_a.pt out_backbone/ --model-id backbone_a --split-id full --seed 0 --epoch 198 --init-group base_a


key                        shape                      L  chunks  seq  c/filt         mu    sigma
------------------------------------------------------------------------------------------------
layer1.0.conv1.weight      (16, 16, 3, 3)          2304      16    1    1.00 -3.005e-05   0.0817
layer1.0.conv2.weight      (16, 16, 3, 3)          2304      16    1    1.00 -3.228e-03   0.0763
layer1.1.conv1.weight      (16, 16, 3, 3)          2304      16    1    1.00 -4.293e-03   0.0706
layer1.1.conv2.weight      (16, 16, 3, 3)          2304      16    1    1.00  1.165e-03   0.0635
layer1.2.conv1.weight      (16, 16, 3, 3)          2304      16    1    1.00 -6.194e-03   0.0641
layer1.2.conv2.weight      (16, 16, 3, 3)          2304      16    1    1.00 -6.340e-03   0.0581
layer2.0.conv1.weight      (32, 16, 3, 3)          4608      32    2    1.00 -1.090e-04   0.0782
layer2.0.conv2.weight      (32, 32, 3, 3)          9216      64    4    2.00 -3.660e-03   0.0654
layer2.1.conv1.weight      (3

### A Zoo of ResNet

In [ ]:
'''group indices by class block, expert k owns classes 20k .. 20k+19'''
zoo_train_feed = {i: [] for i in range(5)}
zoo_test_feed  = {i: [] for i in range(5)}

for idx, label in enumerate(train_dataset.targets):
    zoo_train_feed[label // 20].append(idx)

for idx, label in enumerate(test_dataset.targets):
    zoo_test_feed[label // 20].append(idx)


@torch.no_grad()
def evaluate_grouped(model, loader, expert_id):
    '''accuracy inside the expert's own 20 classes vs everything else'''
    model.eval()
    lo, hi = expert_id * 20, expert_id * 20 + 20
    own_c = own_n = oth_c = oth_n = 0
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        pred = model(images).argmax(1)
        mask = (labels >= lo) & (labels < hi)
        own_c += (pred[mask] == labels[mask]).sum().item()
        own_n += mask.sum().item()
        oth_c += (pred[~mask] == labels[~mask]).sum().item()
        oth_n += (~mask).sum().item()
    return (100.0 * own_c / max(own_n, 1),
            100.0 * oth_c / max(oth_n, 1),
            100.0 * (own_c + oth_c) / (own_n + oth_n))

In [ ]:
def train_expert_model(expert_id, backbone_path='./res_models/backbone_a.pt',
                       seed=SEED, epochs=40, lr=0.01, ckpt_every=4):

    '''seed per expert so runs are reproducible and independently varied'''
    torch.manual_seed(seed + expert_id)
    np.random.seed(seed + expert_id)

    outdir = Path(f'./res_models/expert{expert_id}_seed{seed}')
    outdir.mkdir(parents=True, exist_ok=True)

    '''train only on this expert's 20 classes, labels stay 0-99'''
    train_subset = torch.utils.data.Subset(train_dataset, zoo_train_feed[expert_id])
    train_loader = DataLoader(train_subset, batch_size=128, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)

    '''evaluate on the FULL 100 class test set, that is where specialization shows'''
    full_test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False, num_workers=4, pin_memory=True)

    '''fork the backbone, this is what makes init_group meaningful'''
    expert_model = resnet20(num_classes=100).to(device)
    ckpt = torch.load(backbone_path, map_location=device)
    expert_model.load_state_dict(ckpt['state_dict'], strict=True)

    '''lr 0.01 , high lr erases the backbone and breaks the shared basin'''
    optimizer = optim.SGD(expert_model.parameters(), lr=lr, momentum=0.9,
                          weight_decay=5e-4, nesterov=True)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        expert_model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(expert_model(images), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        scheduler.step()
        train_loss = running_loss / len(train_loader)

        '''checkpoint on a schedule, these accumulate as vae training data'''
        if (epoch + 1) % ckpt_every == 0 or epoch + 1 == epochs:
            own, other, overall = evaluate_grouped(expert_model, full_test_loader, expert_id)
            print(f'  expert {expert_id} ep {epoch+1:3d}/{epochs}  loss {train_loss:.4f}  '
                  f'own {own:.2f}%  other {other:.2f}%  all {overall:.2f}%')
            torch.save({
                'state_dict': expert_model.state_dict(),
                'model_id': f'split{expert_id}_seed{seed}_ep{epoch+1}',
                'split_id': str(expert_id),
                'seed': seed,
                'epoch': epoch + 1,
                'init_group': 'base_a',
                'acc_own': own, 'acc_other': other, 'acc_all': overall,
            }, outdir / f'ep{epoch+1:03d}.pt')

    '''specialization check, the experiment is vacuous without it'''
    own, other, overall = evaluate_grouped(expert_model, full_test_loader, expert_id)
    if own < 70.0:
        print(f'  WARNING expert {expert_id}: own-group {own:.2f}% is low, undertrained')
    if other > 20.0:
        print(f'  WARNING expert {expert_id}: other-group {other:.2f}% too high, did not specialize')
    if other < 1.0:
        print(f'  WARNING expert {expert_id}: other-group {other:.2f}% too low, catastrophic forgetting')
    return own, other, overall

In [13]:
'''run all five, ~10 checkpoints each gives ~50 models for the vae'''
for expert_id in range(5):
    print(f'---- expert {expert_id}, classes {expert_id*20}-{expert_id*20+19} ----')
    train_expert_model(expert_id)

---- expert 0, classes 0-19 ----
  expert 0 ep   4/40  loss 0.1878  own 82.15%  other 16.20%  all 29.39%
  expert 0 ep   8/40  loss 0.1174  own 82.75%  other 10.30%  all 24.79%
  expert 0 ep  12/40  loss 0.0755  own 84.00%  other 7.81%  all 23.05%
  expert 0 ep  16/40  loss 0.0546  own 84.30%  other 6.64%  all 22.17%
  expert 0 ep  20/40  loss 0.0343  own 84.70%  other 5.49%  all 21.33%
  expert 0 ep  24/40  loss 0.0289  own 85.75%  other 5.15%  all 21.27%
  expert 0 ep  28/40  loss 0.0235  own 86.25%  other 4.94%  all 21.20%
  expert 0 ep  32/40  loss 0.0190  own 86.35%  other 4.72%  all 21.05%
  expert 0 ep  36/40  loss 0.0167  own 86.20%  other 4.69%  all 20.99%
  expert 0 ep  40/40  loss 0.0164  own 86.45%  other 4.84%  all 21.16%
---- expert 1, classes 20-39 ----
  expert 1 ep   4/40  loss 0.1392  own 85.25%  other 18.15%  all 31.57%
  expert 1 ep   8/40  loss 0.0910  own 85.85%  other 11.49%  all 26.36%
  expert 1 ep  12/40  loss 0.0564  own 86.95%  other 7.47%  all 23.37%
  expe